# 3a - Modelli Matematici

Questo notebook implementa e ottimizza due classificatori per la predizione
della variabile target `blueWins`:

1. Regressione Logistica;
2. Support Vector Machine (SVM).

I modelli vengono addestrati esclusivamente sul Training Set standardizzato
prodotto durante la fase di Data Preparation.

La selezione degli iperparametri viene effettuata mediante Grid Search con
5-Fold Cross-Validation, mantenendo il Test Set completamente separato per
la successiva valutazione finale.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn import set_config
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.svm import SVC

set_config(display="text")

In [2]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Keep Out\File\Magistrale\predictLoL


## 1. Caricamento del Training Set

Vengono caricati esclusivamente il Training Set standardizzato e la
corrispondente variabile target prodotti durante la fase di Data Preparation.

Il Test Set non viene utilizzato in questa fase, in modo da mantenerlo
completamente indipendente per la valutazione finale dei modelli.

In [3]:
X_TRAIN_PATH = PROJECT_ROOT / "data" / "X_train_scaled.csv"
Y_TRAIN_PATH = PROJECT_ROOT / "data" / "y_train.csv"

assert X_TRAIN_PATH.exists(), f"File non trovato: {X_TRAIN_PATH}"
assert Y_TRAIN_PATH.exists(), f"File non trovato: {Y_TRAIN_PATH}"

X_train = pd.read_csv(X_TRAIN_PATH)
y_train = pd.read_csv(Y_TRAIN_PATH)["blueWins"]

print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")

X_train: (7903, 28)
y_train: (7903,)


In [4]:
print(f"Valori mancanti in X_train: {X_train.isna().sum().sum()}")
print(f"Valori mancanti in y_train: {y_train.isna().sum()}")

print("\nDistribuzione della variabile target:")
print(y_train.value_counts())

Valori mancanti in X_train: 0
Valori mancanti in y_train: 0

Distribuzione della variabile target:
blueWins
0    3959
1    3944
Name: count, dtype: int64


## 2. Regressione Logistica

La Regressione Logistica viene utilizzata come baseline lineare.

Gli iperparametri vengono ottimizzati mediante Grid Search con
5-Fold Cross-Validation.

Vengono confrontati:
- diversi valori del parametro di regolarizzazione `C`;
- penalità L1 e L2.

In [5]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [6]:
logistic_model = LogisticRegression(
    solver="liblinear",
    max_iter=2000,
    random_state=42
)

logistic_param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "l1_ratio": [0, 1]
}

logistic_grid = GridSearchCV(
    estimator=logistic_model,
    param_grid=logistic_param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

logistic_grid.fit(X_train, y_train)

print("Grid Search completata.")

Grid Search completata.


In [7]:
print("Migliori iperparametri:")
print(logistic_grid.best_params_)

best_penalty = (
    "L1"
    if logistic_grid.best_params_["l1_ratio"] == 1
    else "L2"
)

print(f"Penalità selezionata: {best_penalty}")

print(
    f"Migliore accuracy media in Cross-Validation: "
    f"{logistic_grid.best_score_:.4f}"
)

Migliori iperparametri:
{'C': 0.01, 'l1_ratio': 1}
Penalità selezionata: L1
Migliore accuracy media in Cross-Validation: 0.7359


In [8]:
best_logistic_model = logistic_grid.best_estimator_

In [9]:
best_logistic_model

LogisticRegression(C=0.01, l1_ratio=1, max_iter=2000, random_state=42,
                   solver='liblinear')

In [10]:
coefficients = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": best_logistic_model.coef_[0]
})

coefficients["abs_coefficient"] = coefficients["coefficient"].abs()
coefficients["odds_ratio"] = np.exp(coefficients["coefficient"])

coefficients = coefficients.sort_values(
    by="abs_coefficient",
    ascending=False
)

bias = best_logistic_model.intercept_[0]

print(f"Bias (b): {bias:.12f}")

coefficients

Bias (b): 0.000000000000


,feature,coefficient,abs_coefficient,odds_ratio
14,blueGoldDiff,0.930005,0.930005,2.534522
15,blueExperienceDiff,0.392365,0.392365,1.480477
6,blueDragons,0.128562,0.128562,1.137192
19,redDragons,-0.108082,0.108082,0.897554
3,blueFirstBlood,0.000000,0.000000,1.000000
4,blueKills,0.000000,0.000000,1.000000
5,blueDeaths,0.000000,0.000000,1.000000
7,blueHeralds,0.000000,0.000000,1.000000
8,blueTowersDestroyed,0.000000,0.000000,1.000000
0,gameId,0.000000,0.000000,1.000000


### Interpretazione dei coefficienti

La Regressione Logistica calcola inizialmente una combinazione lineare delle feature:

$$
z = b + w_1x_1 + w_2x_2 + \dots + w_nx_n
$$

dove $w_i$ rappresenta il coefficiente associato alla feature $x_i$ e $b$ rappresenta il termine di bias.

Il valore ottenuto viene trasformato in una probabilità attraverso la funzione sigmoide:

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$

ottenendo:

$$
P(\text{blueWins} = 1 \mid X) = \sigma(z)
$$

Un coefficiente positivo aumenta il logaritmo delle odds di vittoria del Blue Team e, di conseguenza, aumenta la probabilità prevista di `blueWins = 1`.

Un coefficiente negativo produce invece l'effetto opposto.

Poiché le feature sono state standardizzate tramite Z-Score, una variazione unitaria di una feature corrisponde a una variazione di una deviazione standard rispetto alla media del Training Set. I coefficienti risultano quindi maggiormente confrontabili tra loro.

L'esponenziale del coefficiente, $e^{w_i}$, rappresenta inoltre l'odds ratio associato all'aumento di una deviazione standard della corrispondente feature, mantenendo costanti le altre variabili.

## 3. Support Vector Machine

Il secondo modello utilizzato è una Support Vector Machine.

La Grid Search confronta:
- kernel lineare;
- kernel RBF;

per differenti valori del parametro di penalizzazione `C`.

Anche in questo caso la selezione degli iperparametri viene effettuata
mediante 5-Fold Cross-Validation sul solo Training Set.

In [11]:
svm_model = SVC()

svm_param_grid = {
    "kernel": ["linear", "rbf"],
    "C": [0.01, 0.1, 1, 10, 100]
}

svm_grid = GridSearchCV(
    estimator=svm_model,
    param_grid=svm_param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

svm_grid.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=SVC(), n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 10, 100],
                         'kernel': ['linear', 'rbf']},
             scoring='accuracy')

In [12]:
print("Migliori iperparametri:")
print(svm_grid.best_params_)

print(f"\nMigliore accuracy media in Cross-Validation: "
      f"{svm_grid.best_score_:.4f}")

Migliori iperparametri:
{'C': 0.01, 'kernel': 'linear'}

Migliore accuracy media in Cross-Validation: 0.7335


In [13]:
best_svm_model = svm_grid.best_estimator_

### Importanza della standardizzazione per SVM

La standardizzazione delle feature è particolarmente importante per le
Support Vector Machine, poiché la costruzione dell'iperpiano a massimo
margine dipende dalla geometria dello spazio delle feature.

Se le variabili presentassero scale numeriche molto differenti, le feature
caratterizzate da valori numericamente più grandi avrebbero un'influenza
sproporzionata sul calcolo delle distanze e dell'iperpiano di separazione.

Attraverso la standardizzazione Z-Score,

$$
z = \frac{x-\mu}{\sigma}
$$

ogni feature viene espressa rispetto alla propria media e deviazione
standard, rendendo le diverse dimensioni maggiormente confrontabili.

Questo aspetto è particolarmente rilevante anche per il kernel RBF, che
dipende dalla distanza tra le osservazioni nello spazio delle feature.
Senza scaling, una variabile caratterizzata da una scala numerica elevata
potrebbe dominare tale distanza e quindi influenzare in maniera
sproporzionata la funzione kernel.

Per evitare data leakage, media e deviazione standard utilizzate per
lo scaling sono state calcolate esclusivamente sul Training Set durante
la fase di Data Preparation.

## 4. Salvataggio dei modelli

I due classificatori ottimizzati vengono salvati nella cartella `models/`
in formato `.pkl` mediante `joblib`, in modo da poter essere caricati
successivamente senza effettuare nuovamente l'addestramento.

In [14]:
MODELS_DIR = PROJECT_ROOT / "models"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

LOGISTIC_MODEL_PATH = MODELS_DIR / "logistic_regression.pkl"
SVM_MODEL_PATH = MODELS_DIR / "svm.pkl"

joblib.dump(best_logistic_model, LOGISTIC_MODEL_PATH)
joblib.dump(best_svm_model, SVM_MODEL_PATH)

print(f"Regressione Logistica salvata in: {LOGISTIC_MODEL_PATH}")
print(f"SVM salvata in: {SVM_MODEL_PATH}")

Regressione Logistica salvata in: c:\Keep Out\File\Magistrale\predictLoL\models\logistic_regression.pkl
SVM salvata in: c:\Keep Out\File\Magistrale\predictLoL\models\svm.pkl


## 5. Controllo finale


In [15]:
print("Controllo file salvati:")

print(
    f"{'OK' if LOGISTIC_MODEL_PATH.exists() else 'ERRORE'} "
    f"- {LOGISTIC_MODEL_PATH.name}"
)

print(
    f"{'OK' if SVM_MODEL_PATH.exists() else 'ERRORE'} "
    f"- {SVM_MODEL_PATH.name}"
)

Controllo file salvati:
OK - logistic_regression.pkl
OK - svm.pkl
